# Mission 2: Digit Reader from Scratch 🔢

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anshid/MathMeetsML/blob/main/notebooks/02_digit_reader.ipynb)

> **From your syllabus**
> - *ML-I*, Unit III (logistic regression) and Unit II (regularised least squares, bias-variance)
> - *Convex Optimization*, Unit II (convexity) and Unit V (linear classifiers)

In Mission 0 a library did the work. Now **you** build the digit reader with nothing but NumPy and the gradient descent you wrote in Mission 1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

np.random.seed(0)
digits = load_digits()
X, y = digits.data / 16.0, digits.target     # scale pixels to [0, 1]
order = np.random.permutation(len(X))
train, test = order[:1300], order[1300:]
print(X[train].shape, X[test].shape)

## 1. Warm-up: is it a 0 or a 1?
Two classes first. Label $t_i=1$ if the digit is 1, else 0. The model gives the probability
$$p_i=\sigma(w^\top x_i+b),\qquad \sigma(z)=\frac1{1+e^{-z}}.$$
**Core 1:** fill in the sigmoid.

<details><summary>Hint</summary>

`np.exp(-z)` is $e^{-z}$.
</details>

In [ ]:
def sigmoid(z):
    return ____            # TODO: 1 / (1 + e^{-z})

print(sigmoid(np.array([-5.0, 0.0, 5.0])))   # expect about [0.007 0.5 0.993]

## 2. The loss and its gradient
Cross-entropy loss (ML-I Unit III), averaged over $n$ examples:
$$L(w,b)=-\frac1n\sum_i\big[t_i\log p_i+(1-t_i)\log(1-p_i)\big]$$

> **Math box.** Using $\sigma'(z)=\sigma(1-\sigma)$, the chain rule collapses beautifully to
> $$\nabla_w L=\frac1n X^\top(p-t),\qquad \frac{\partial L}{\partial b}=\frac1n\sum_i(p_i-t_i).$$
> $L$ is convex in $(w,b)$ (its Hessian $\frac1nX^\top\mathrm{diag}(p(1-p))X\succeq0$), so gradient descent cannot get stuck in a bad valley (Convex Optimization Unit II).

**Core 2:** fill in the gradient, then let the computer check your calculus.

In [ ]:
def loss(w, b, X, t):
    p = sigmoid(X @ w + b)
    eps = 1e-12
    return -np.mean(t * np.log(p + eps) + (1 - t) * np.log(1 - p + eps))

def gradient(w, b, X, t):
    p = sigmoid(X @ w + b)
    grad_w = ____          # TODO: (1/n) X^T (p - t)
    grad_b = ____          # TODO: mean of (p - t)
    return grad_w, grad_b

# Finite-difference check on random data
Xr, tr, wr = np.random.randn(20, 5), (np.random.rand(20) > 0.5) * 1.0, np.random.randn(5)
gw, gb = gradient(wr, 0.3, Xr, tr)
num = np.array([(loss(wr + 1e-5 * e, 0.3, Xr, tr) - loss(wr - 1e-5 * e, 0.3, Xr, tr)) / 2e-5 for e in np.eye(5)])
assert np.allclose(gw, num, atol=1e-6), "Gradient mismatch - recheck X^T (p - t) / n"
print("Gradient correct ✔")

**Core 3:** the training loop. Same golf swing as Mission 1: step against the gradient.

In [ ]:
def fit(X, t, lr=1.0, steps=300):
    w, b = np.zeros(X.shape[1]), 0.0
    history = []
    for _ in range(steps):
        gw, gb = gradient(w, b, X, t)
        w = ____               # TODO: gradient descent update for w
        b = ____               # TODO: gradient descent update for b
        history.append(loss(w, b, X, t))
    return w, b, history

mask = (y[train] <= 1)
Xtr, ttr = X[train][mask], (y[train][mask] == 1) * 1.0
w, b, hist = fit(Xtr, ttr)
plt.plot(hist); plt.xlabel("stroke"); plt.ylabel("loss"); plt.show()

mt = (y[test] <= 1)
pred = sigmoid(X[test][mt] @ w + b) > 0.5
print("0-vs-1 test accuracy:", np.mean(pred == (y[test][mt] == 1)))

## 3. What did it learn?
The weight vector has 64 entries, one per pixel. Reshape it back to $8\times8$: red pixels say "this is a 1", blue say "this is a 0".

In [ ]:
plt.imshow(w.reshape(8, 8), cmap="coolwarm"); plt.colorbar(); plt.title("learned weights"); plt.show()

## 4. All ten digits: softmax
With 10 classes use one weight vector per class and $p_k=\dfrac{e^{z_k}}{\sum_j e^{z_j}}$. The gradient has the same shape:
$\nabla_W L=\frac1nX^\top(P-T)$ where $T$ is the one-hot label matrix. The code is provided; read it and notice how little changed.

In [ ]:
def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)      # subtract the max for numerical stability
    E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)

def fit_softmax(X, labels, lr=1.0, steps=300, lam=0.0):
    T = np.eye(10)[labels]
    W, b = np.zeros((X.shape[1], 10)), np.zeros(10)
    for _ in range(steps):
        P = softmax(X @ W + b)
        W -= lr * (X.T @ (P - T) / len(X) + lam * W)
        b -= lr * (P - T).mean(axis=0)
    return W, b

def accuracy(W, b, X, labels):
    return np.mean(np.argmax(X @ W + b, axis=1) == labels)

W, b = fit_softmax(X[train], y[train])
print("train accuracy:", round(accuracy(W, b, X[train], y[train]), 3))
print("test accuracy :", round(accuracy(W, b, X[test], y[test]), 3))

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4.2))
for k, ax in enumerate(axes.ravel()):
    ax.imshow(W[:, k].reshape(8, 8), cmap="coolwarm"); ax.axis("off"); ax.set_title(f"digit {k}")
plt.tight_layout(); plt.show()

## Stretch: regularisation and bias-variance
Add the penalty $\frac\lambda2\|W\|^2$ (ridge, *ML-I Unit II*). Its gradient is $\lambda W$, already built into `fit_softmax` via `lam`.
Train on only 100 digits: train accuracy hits 100% but test lags, a sign of overfitting. Sweep $\lambda$ and **report honestly**: does it help here? Real experiments sometimes say no. Why might it not? (Think: how much data, how many steps; stopping early is also a form of regularisation.)

In [ ]:
small = train[:100]
for lam in [0.0, 0.003, 0.01, 0.03]:
    W2, b2 = fit_softmax(X[small], y[small], lam=lam, steps=500)
    print(f"lambda={lam:<6} train={accuracy(W2, b2, X[small], y[small]):.3f}  test={accuracy(W2, b2, X[test], y[test]):.3f}")

## Quest (take-home)
Plot the **confusion matrix** of your classifier (10x10 counts of true vs predicted). Which two digits get confused most, and why does their weight image explain it? Show the 12 most confidently wrong pictures.

## You just used...
- the **sigmoid/softmax** model and **cross-entropy** loss (ML-I Unit III)
- the **chain rule** to get $\nabla L=\frac1nX^\top(p-t)$, and a finite-difference check
- **convexity** to trust gradient descent (Convex Optimization Unit II)
- **ridge regularisation** and the bias-variance trade-off (ML-I Unit II)

## Your CV line
> *Implemented multinomial logistic regression from scratch in NumPy (gradient derived and verified numerically), reaching about 95% test accuracy on handwritten digits; studied L2 regularisation and overfitting.*